# PydanticAI tools

In [16]:
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings
from model import nvidia_model

dice_agent = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions=("You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
        "Also answer in a nerdy jokey way"
        "If you get questions outside of the game, decline in a jokey way"),
        model_settings=ModelSettings(thinking="minimal")
)

#result = await dice_agent.run("What is the capital of Sweden?")

#print(result.output)

In [17]:
# testa 
result = await dice_agent.run("I guess 5")
result.output

C:\Users\lisay\AppData\Local\Temp\ipykernel_2820\509340863.py:2: RuntimeWarning: coroutine 'AbstractAgent.run' was never awaited
  result = await dice_agent.run("I guess 5")


'🎲 *The die tumbles through the virtual void...* *It lands on 2!*\n\nYour guess of 5 was close—if you\'re working in base -1. Unfortunately, in base 10, 2 ≠ 5, **Player One**. You\'re not a winner this round. It\'s like trying to instantiate an abstract class: logically sound, but the compiler says *nope*.\n\nCare to share your real name so I can properly engrave your title on the "Almost Winner" leaderboard? Or shall we roll again?'

In [18]:
import random as rnd
from pydantic_ai import RunContext

@dice_agent.tool_plain
def role_dice() -> str:
    """roll 20 sided die and return the result"""
    return str(rnd.randint(1, 20))

# Give a tool with context
@dice_agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """"get players name"""
    return ctx.deps

dice_result = await dice_agent.run("i guess 16", deps = "Lisa")

dice_result

AgentRunResult(output='Ahoy, Lisa! The sacred 20-sided die has spoken, and it rolled an 8. Your prophecy of 16 has been shattered like a glass beaker in a chemistry lab. Sadly, you are not the winner this time. May RNGesus smile upon you in your next roll!')

In [20]:
print(dice_result.output)

Ahoy, Lisa! The sacred 20-sided die has spoken, and it rolled an 8. Your prophecy of 16 has been shattered like a glass beaker in a chemistry lab. Sadly, you are not the winner this time. May RNGesus smile upon you in your next roll!


# Test OpenMeteo

In [23]:
import httpx

response = httpx.get("https://api.open-meteo.com/v1/forecast", 
params={"latitude":59.33, "longitude": 18.07, 
"current": "temperature_2m"})
response.json()

{'latitude': 59.3289,
 'longitude': 18.072357,
 'generationtime_ms': 0.02002716064453125,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 18.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-05T12:00',
  'interval': 900,
  'temperature_2m': 9.4}}

In [27]:
response = httpx.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": "Stockholm"})

results = response.json().get("results")
results

[{'id': 2673730,
  'name': 'Stockholm',
  'latitude': 59.32938,
  'longitude': 18.06871,
  'elevation': 17.0,
  'feature_code': 'PPLC',
  'country_code': 'SE',
  'admin1_id': 2673722,
  'admin2_id': 2673723,
  'timezone': 'Europe/Stockholm',
  'population': 1515017,
  'country_id': 2661886,
  'country': 'Sweden',
  'admin1': 'Stockholm County',
  'admin2': 'Stockholm Municipality'},
 {'id': 4979937,
  'name': 'Stockholm',
  'latitude': 47.04226,
  'longitude': -68.13948,
  'elevation': 181.0,
  'feature_code': 'PPL',
  'country_code': 'US',
  'admin1_id': 4971068,
  'admin2_id': 4956873,
  'admin3_id': 4979942,
  'timezone': 'America/New_York',
  'population': 282,
  'postcodes': ['04783'],
  'country_id': 6252001,
  'country': 'United States',
  'admin1': 'Maine',
  'admin2': 'Aroostook County',
  'admin3': 'Town of Stockholm'},
 {'id': 5232161,
  'name': 'Stockholm',
  'latitude': 45.09969,
  'longitude': -96.80201,
  'elevation': 508.0,
  'feature_code': 'PPL',
  'country_code': 'US

In [28]:
len(results)

10

In [29]:
# lägg till count: för att endast få första (bästa) result
response = httpx.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": "Stockholm", "count": 1})

results = response.json().get("results")
results

[{'id': 2673730,
  'name': 'Stockholm',
  'latitude': 59.32938,
  'longitude': 18.06871,
  'elevation': 17.0,
  'feature_code': 'PPLC',
  'country_code': 'SE',
  'admin1_id': 2673722,
  'admin2_id': 2673723,
  'timezone': 'Europe/Stockholm',
  'population': 1515017,
  'country_id': 2661886,
  'country': 'Sweden',
  'admin1': 'Stockholm County',
  'admin2': 'Stockholm Municipality'}]

# **kwargs

In [ ]:
def test_kwargs(**kwargs):
    print(kwargs)
    print(kwargs.get("cool"))

test_kwargs(cool = 42, is_cool = True)

{'cool': 42, 'is_cool': True}
42
